In [1]:
# install albumentations if not present
#!pip install -q albumentations

import os, cv2, numpy as np
import albumentations as A
from glob import glob
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
print("TF version:", tf.__version__)


TF version: 2.19.0


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# optional: restrict GPU memory growth (helps avoid OOM)
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except Exception as e:
        print("GPU config error:", e)
else:
    print("No GPU found")

# set dataset root (adjust if your path differs)
DATASET_ROOT = "/content/drive/MyDrive/Project Data/Fruit"
TRAIN_ROOT = f"{DATASET_ROOT}/Train"
VAL_ROOT   = f"{DATASET_ROOT}/Validation"

import os
print("TRAIN exists:", os.path.exists(TRAIN_ROOT))
print("VAL exists:  ", os.path.exists(VAL_ROOT))


Mounted at /content/drive
GPU memory growth enabled
TRAIN exists: True
VAL exists:   True


In [3]:
# Fruit class folders are inside TRAIN_ROOT
fruit_classes = sorted([
    d for d in os.listdir(TRAIN_ROOT)
    if os.path.isdir(os.path.join(TRAIN_ROOT, d))
])
if len(fruit_classes) == 0:
    raise RuntimeError("No fruit subfolders found under TRAIN_ROOT. Check path.")

num_classes = len(fruit_classes) + 1  # + background
print("Detected classes:", fruit_classes)
print("Num classes (including background):", num_classes)

# map folder -> id (fruits:1..N, background:0)
class2id = {fruit: i+1 for i, fruit in enumerate(fruit_classes)}
class2id["background"] = 0

# Unique colors per fruit (R,G,B). Deterministic using seed.
np.random.seed(42)
class_colors = { class2id[fruit]: np.random.randint(0,255,3).tolist() for fruit in fruit_classes }
class_colors[0] = [0,0,0]   # background black

# quick print
print("class2id sample:", list(class2id.items())[:6])
print("class_colors sample:", list(class_colors.items())[:6])


Detected classes: ['Apple_Gala', 'Apple_Golden Delicious', 'Avocado', 'Banana', 'Berry', 'Burmese Grape', 'Carambola', 'Date Palm', 'Dragon', 'Elephant Apple', 'Grape', 'Green Coconut', 'Guava', 'Hog Plum', 'Kiwi', 'Lichi', 'Malta', 'Mango Golden Queen', 'Mango_Alphonso', 'Mango_Amrapali', 'Mango_Bari', 'Mango_Himsagar', 'Olive', 'Orange', 'Palm', 'Persimmon', 'Pineapple', 'Pomegranate', 'Watermelon', 'White Pear']
Num classes (including background): 31
class2id sample: [('Apple_Gala', 1), ('Apple_Golden Delicious', 2), ('Avocado', 3), ('Banana', 4), ('Berry', 5), ('Burmese Grape', 6)]
class_colors sample: [(1, [102, 179, 92]), (2, [14, 106, 71]), (3, [188, 20, 102]), (4, [121, 210, 214]), (5, [74, 202, 87]), (6, [116, 99, 103])]


In [4]:
# DO NOT MODIFY — your exact functions
def preprocess_image(img, target_size=(224,224)):
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img = cv2.resize(img, target_size)

    # sharpening
    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    img = cv2.filter2D(img, -1, kernel)

    return img.astype("float32")/255.0

def get_augmentation_pipeline(size=(224,224)):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.RandomResizedCrop(size=size, scale=(0.8,1.0), ratio=(0.9,1.1), p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5),
        A.RandomBrightnessContrast(0.3,0.3,p=0.5)
    ])


In [5]:
def list_files(folder):
    return sorted(glob(os.path.join(folder, "*.*")))

def build_pairs(root_folder):
    pairs = []
    for fruit in fruit_classes:
        img_dir  = os.path.join(root_folder, fruit, "Images")
        mask_dir = os.path.join(root_folder, fruit, "Mask")
        if not os.path.isdir(img_dir) or not os.path.isdir(mask_dir):
            # try lower-case or other common names fallback
            img_dir = os.path.join(root_folder, fruit, "images")
            mask_dir = os.path.join(root_folder, fruit, "masks")
        img_paths  = list_files(img_dir)
        mask_paths = list_files(mask_dir)
        if len(img_paths) == 0 or len(mask_paths) == 0:
            print(f"Warning: no files for {fruit} in {img_dir} or {mask_dir}")
            continue
        # zip (pairs) — if counts mismatch we zip to min length
        for im_p, m_p in zip(img_paths, mask_paths):
            pairs.append((im_p, m_p, class2id[fruit]))
    return pairs

train_pairs = build_pairs(TRAIN_ROOT)
val_pairs = build_pairs(VAL_ROOT)

print("Train pairs:", len(train_pairs))
print("Val pairs:  ", len(val_pairs))
if len(train_pairs) == 0 or len(val_pairs) == 0:
    raise RuntimeError("No train/val pairs found. Check folder names and files.")


Train pairs: 1759
Val pairs:   150


In [6]:
import math
TARGET_SIZE = (224,224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
augmenter = get_augmentation_pipeline(size=TARGET_SIZE)

def batch_generator(pairs, batch_size=BATCH_SIZE, augment=False, shuffle=True):
    # infinite generator for tf.data.from_generator
    while True:
        if shuffle:
            np.random.shuffle(pairs)
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            imgs = []
            masks = []
            for img_path, mask_path, cid in batch:
                img = cv2.imread(img_path)
                mask = cv2.imread(mask_path, 0)
                if img is None or mask is None:
                    continue
                # Resize FIRST (fix shape mismatch)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_LINEAR)
                mask = cv2.resize(mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
                # Convert mask to class id
                mask = np.where(mask>0, cid, 0).astype(np.uint8)
                # Augment after resize
                if augment:
                    try:
                        d = augmenter(image=img, mask=mask)
                        img, mask = d["image"], d["mask"]
                    except Exception as e:
                        # if augmentation fails, skip this sample
                        print("Augment error:", e)
                        continue
                # final preprocess (your function)
                img = preprocess_image(img, target_size=TARGET_SIZE)
                mask = to_categorical(mask, num_classes=num_classes)
                imgs.append(img)
                masks.append(mask)
            if len(imgs) == 0:
                continue
            imgs = np.stack(imgs, axis=0).astype(np.float32)
            masks = np.stack(masks, axis=0).astype(np.float32)
            yield imgs, masks

# Create tf.data datasets from python generator
output_types = (tf.float32, tf.float32)
# shapes are dynamic because last batch may be smaller; use None for batch dim
output_shapes = (tf.TensorShape([None, TARGET_SIZE[0], TARGET_SIZE[1], 3]),
                 tf.TensorShape([None, TARGET_SIZE[0], TARGET_SIZE[1], num_classes]))

train_ds = tf.data.Dataset.from_generator(
    lambda: batch_generator(train_pairs, batch_size=BATCH_SIZE, augment=True, shuffle=True),
    output_signature=(tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], 3), dtype=tf.float32),
                      tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], num_classes), dtype=tf.float32))
)

val_ds = tf.data.Dataset.from_generator(
    lambda: batch_generator(val_pairs, batch_size=BATCH_SIZE, augment=False, shuffle=False),
    output_signature=(tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], 3), dtype=tf.float32),
                      tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], num_classes), dtype=tf.float32))
)

# Prefetch to improve performance
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# number of steps per epoch
steps_per_epoch = math.ceil(len(train_pairs) / BATCH_SIZE)
validation_steps = math.ceil(len(val_pairs) / BATCH_SIZE)
print("steps_per_epoch:", steps_per_epoch, "validation_steps:", validation_steps)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


steps_per_epoch: 220 validation_steps: 19


In [7]:
def conv_block(x,f):
    x = layers.Conv2D(f,3,padding="same",activation="relu")(x)
    x = layers.Conv2D(f,3,padding="same",activation="relu")(x)
    return x

def build_unet(input_shape=(224,224,3), nclasses=num_classes):
    inp = layers.Input(input_shape)
    c1 = conv_block(inp,32); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1,64);  p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2,128); p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3,256); p4 = layers.MaxPooling2D()(c4)
    bn = conv_block(p4,512)
    u4 = layers.UpSampling2D()(bn); u4 = layers.concatenate([u4,c4]); c5 = conv_block(u4,256)
    u3 = layers.UpSampling2D()(c5); u3 = layers.concatenate([u3,c3]); c6 = conv_block(u3,128)
    u2 = layers.UpSampling2D()(c6); u2 = layers.concatenate([u2,c2]); c7 = conv_block(u2,64)
    u1 = layers.UpSampling2D()(c7); u1 = layers.concatenate([u1,c1]); c8 = conv_block(u1,32)
    out = layers.Conv2D(nclasses,1,activation="softmax")(c8)
    return models.Model(inp,out)

model = build_unet()
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1, num_classes])
    y_pred_f = tf.reshape(y_pred, [-1, num_classes])
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denom = tf.reduce_sum(y_true_f + y_pred_f, axis=0)
    dice = (2. * intersection + smooth) / (denom + smooth)
    return 1 - tf.reduce_mean(dice)

def combined_loss(y_true, y_pred):
    ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    dl = dice_loss(y_true, y_pred)
    return ce + dl

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │     36,928 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 56, 56,    │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 28, 28,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 28, 28,    │    590,080 │ conv2d_6[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 14, 14,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 14, 14,    │  1,180,160 │ max_pooling2d_3[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 14, 14,    │  2,359,808 │ conv2d_8[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 28, 28,    │          0 │ conv2d_9[0][0]    │
│ (UpSampling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 28, 28,    │          0 │ up_sampling2d[0]

 Total params: 7,847,647 (29.94 MB)

 Trainable params: 7,847,647 (29.94 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)


In [9]:
EPOCHS = 80

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)



Epoch 1/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 1789s 8s/step - accuracy: 0.6014 - loss: 3.1756 - val_accuracy: 0.6261 - val_loss: 2.5386 - learning_rate: 1.0000e-04
Epoch 2/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 125s 569ms/step - accuracy: 0.6283 - loss: 2.5905 - val_accuracy: 0.6261 - val_loss: 2.4545 - learning_rate: 1.0000e-04
Epoch 3/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 82s 372ms/step - accuracy: 0.6282 - loss: 2.5443 - val_accuracy: 0.6261 - val_loss: 2.4607 - learning_rate: 1.0000e-04
Epoch 4/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 78s 356ms/step - accuracy: 0.6213 - loss: 2.5056 - val_accuracy: 0.6391 - val_loss: 2.3502 - learning_rate: 1.0000e-04
Epoch 5/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 344ms/step - accuracy: 0.6415 - loss: 2.3957 - val_accuracy: 0.6478 - val_loss: 2.2571 - learning_rate: 1.0000e-04
Epoch 6/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 71s 323ms/step - accuracy: 0.6522 - loss: 2.2960 - val_accuracy: 0.6646 - val_loss: 2.2331 - learning_rate: 1.0000e-04
Epoch 7/80
220/220 ━━━━━━━━━━━━━━━━━━━━ 75s 34

In [10]:
import numpy as np

# helper to compute metrics over full val set
def evaluate_on_dataset(model, pairs, batch_size=BATCH_SIZE):
    total_pixels = 0
    correct_pixels = 0
    iou_list = []
    # per-class intersection and union
    inter = np.zeros(num_classes, dtype=np.float64)
    union = np.zeros(num_classes, dtype=np.float64)

    gen = batch_generator(pairs, batch_size=batch_size, augment=False, shuffle=False)
    steps = math.ceil(len(pairs)/batch_size)
    for _ in range(steps):
        try:
            Xb, Yb = next(gen)
        except StopIteration:
            break
        preds = model.predict(Xb)
        pred_lbl = np.argmax(preds, axis=-1)
        true_lbl = np.argmax(Yb, axis=-1)
        # pixel accuracy
        correct_pixels += np.sum(pred_lbl == true_lbl)
        total_pixels += np.prod(true_lbl.shape)
        # per-class inter/union
        for c in range(num_classes):
            inter[c] += np.sum((pred_lbl==c) & (true_lbl==c))
            union[c] += np.sum((pred_lbl==c) | (true_lbl==c))
    pixel_acc = correct_pixels / total_pixels
    ious = [ (inter[c]/union[c]) for c in range(num_classes) if union[c] > 0 ]
    mean_iou = np.mean(ious) if len(ious)>0 else 0.0
    return pixel_acc, mean_iou, inter, union

pixel_acc, mean_iou, inter, union = evaluate_on_dataset(model, val_pairs)
print("Pixel accuracy:", pixel_acc)
print("Mean IoU:", mean_iou)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Pixel accuracy: 0.8828410661139455
Mean IoU: 0.6876488117865369


In [11]:
OUT_DIR = "/content/drive/MyDrive/Project Data/Fruit/Predictions_PartE_improved"
os.makedirs(OUT_DIR, exist_ok=True)

# iterate through validation pairs and save predictions side-by-side
gen = batch_generator(val_pairs, batch_size=1, augment=False, shuffle=False)
for idx in range(len(val_pairs)):
    Xb, Yb = next(gen)
    pred = model.predict(Xb)  # shape (1,H,W,C)
    pred_lbl = np.argmax(pred[0], axis=-1)
    true_lbl = np.argmax(Yb[0], axis=-1)

    # render colored masks
    def render_mask(mask_lbl):
        h,w = mask_lbl.shape
        out = np.zeros((h,w,3), dtype=np.uint8)
        for cid, color in class_colors.items():
            out[mask_lbl==cid] = color
        return out

    img_vis = (Xb[0]*255).astype(np.uint8)
    pred_color = render_mask(pred_lbl)
    true_color = render_mask(true_lbl)

    # stack horizontally: image | true | pred
    combined = np.concatenate([img_vis, true_color, pred_color], axis=1)
    save_path = os.path.join(OUT_DIR, f"val_{idx:04d}.png")
    # cv2 expects BGR
    cv2.imwrite(save_path, cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

print("Saved predictions to:", OUT_DIR)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━

In [15]:
# ============================
# SAVE TRAINED MODEL + WEIGHTS
# ============================

SAVE_DIR = "/content/drive/MyDrive/Project Data/Fruit/Trained_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1️⃣ Save full model in NEW Keras format
model.save(os.path.join(SAVE_DIR, "unet_fruit_segmentation.keras"))
print("✔ Full model (.keras format) saved.")

# 2️⃣ Save only weights (must end with .weights.h5)
model.save_weights(os.path.join(SAVE_DIR, "unet_fruit.weights.h5"))
print("✔ Model weights saved (.weights.h5).")

# 3️⃣ Export SavedModel (for TensorFlow / TFLite / TF-Serving)
model.export(os.path.join(SAVE_DIR, "savedmodel_format"))
print("✔ SavedModel exported.")

# 4️⃣ Optional: save training history
import json
with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
    json.dump(history.history, f)

print("🎉 All model files saved to:", SAVE_DIR)


✔ Full model (.keras format) saved.
✔ Model weights saved (.weights.h5).
Saved artifact at '/content/drive/MyDrive/Project Data/Fruit/Trained_Model/savedmodel_format'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 224, 224, 31), dtype=tf.float32, name=None)
Captures:
  139215561358608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557251856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557251664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557252432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557252240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557251472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557249744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139215557251088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1392155572537